# AutoVQA - End-to-End Example

This notebook demonstrates the **complete pipeline** in order:

1. **Collect** — Download raw images & text dataset
2. **Preprocess** — Clean and normalize raw images
3. **Augment** — Generate VQA pairs from preprocessed images using Gemini
4. **Evaluate** — Score each QA pair + image using Gemini (image quality, text quality, VQA correlation)
5. **EDA** — Exploratory data analysis on the evaluated data
6. **Filter** — Quality filtering based on scores
7. **Balance** — Class balancing

> **Important:** Steps 5–7 require the output of Step 4 (`evaluated_vqa.json`).  
> Make sure to configure your Google Cloud / Gemini credentials before running Steps 3–4.

---
## Step 1: Collect

Download the default dataset (text data + images from URLs).

Output:
- `./data/text_extracted/` — extracted text JSON files
- `./data/image_extracted/raw_images_from_urls/` — downloaded images

In [1]:
from autovqa.collect import download_default_data

# Downloads to ./data/ — skips already-downloaded files automatically
download_default_data(output="./data")

2026-07-14 17:57:22.864 | INFO     | autovqa.collect:data_downloader:36 - Downloading data from Google Drive with ID: 1hRXBHR3QAUy5XPbdyBGsge3GH9FQB4x9 to ./data/textzip/Data.zip
Downloading...
From: https://drive.google.com/uc?id=1hRXBHR3QAUy5XPbdyBGsge3GH9FQB4x9
To: /home/ddyln/Documents/School/Code/AutoVQA/docs/data/textzip/Data.zip
100%|██████████| 9.96M/9.96M [00:00<00:00, 24.4MB/s]
2026-07-14 17:57:29.790 | INFO     | autovqa.collect:data_downloader:40 - Download completed.
2026-07-14 17:57:29.790 | INFO     | autovqa.collect:data_extraction:51 - The directory ./data/data_text already exists.
2026-07-14 17:57:29.978 | INFO     | autovqa.collect:data_extraction:57 - Data extraction completed.
2026-07-14 17:57:29.979 | INFO     | autovqa.collect:download_image_from_urls:75 - Found JSON file: datasetQA_combined.json
2026-07-14 17:57:30.149 | INFO     | autovqa.collect:download_image_from_urls:93 - Total unique URLs: 32044
2026-07-14 17:57:30.164 | INFO     | autovqa.collect:download

Error 404 with URL: http://images.cocodataset.org/train2017/000000536.jpg


---
## Step 2: Preprocess

Run the image preprocessing pipeline on the raw downloaded images before augmentation.

Output:
- `./data/processed_images/` — preprocessed images ready for augmentation

In [8]:
from autovqa.preprocess.main import run_pipeline

# Preprocess all raw images before augmentation
run_pipeline(
    input_folder="./data/data_images/raw_images_from_urls",
    output_folder="./data/processed_images",
    do_normalize=False,
)

Scanned 1000/32043 images
Scanned 2000/32043 images
Scanned 3000/32043 images
Scanned 4000/32043 images
Scanned 5000/32043 images
Scanned 6000/32043 images
Scanned 7000/32043 images
Scanned 8000/32043 images
Scanned 9000/32043 images
Scanned 10000/32043 images
Scanned 11000/32043 images
Scanned 12000/32043 images
Scanned 13000/32043 images
Scanned 14000/32043 images
Scanned 15000/32043 images
Scanned 16000/32043 images
Scanned 17000/32043 images
Scanned 18000/32043 images
Scanned 19000/32043 images
Scanned 20000/32043 images
Scanned 21000/32043 images
Scanned 22000/32043 images
Scanned 23000/32043 images
Scanned 24000/32043 images
Scanned 25000/32043 images
Scanned 26000/32043 images
Scanned 27000/32043 images
Scanned 28000/32043 images
Scanned 29000/32043 images
Scanned 30000/32043 images
Scanned 31000/32043 images
Scanned 32000/32043 images
Scanned 32043/32043 images

Most common size: (480x640) - 6938 images
Using target size: (480, 640)
Found 32043 images in ./data/data_images/raw_

KeyboardInterrupt: 

---
## Step 3: Augment

Generate VQA question-answer pairs from the preprocessed images using Gemini.

### Prerequisites
Create a config file at `src/autovqa/augment/config.toml`:

```toml
[gemini]
api_key = "YOUR_GEMINI_API_KEY"
model_name = "models/gemini-flash-lite-latest"
temperature = 0.9

[qa]
wait_time = 10
max_retries = 3
```

Output: `./output/augmented_vqa.json`

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "your-api-key-here"

In [2]:
import json
import os
from pathlib import Path
from autovqa.augment.client import AugmentClient

os.makedirs("./output", exist_ok=True)

# Pick a single image to keep this example quick
image_folder = Path("./data/processed_images")
sample_image = next(f for f in image_folder.iterdir() if f.is_file())

client = AugmentClient(service_name="google")
result = client.generate_response(sample_image)

if result:
    entry = client.format_response(
        json_response=result.model_dump(),
        image_path=str(sample_image)
    )
    with open("./output/augmented_vqa.json", "w", encoding="utf-8") as f:
        json.dump(entry, f, ensure_ascii=False, indent=2)
    print(f"Generated {len(entry)} QA entries -> ./output/augmented_vqa.json")
else:
    print("No response. Check your API key and config.")

2026-07-14 18:50:05.222 | DEBUG    | autovqa.augment.config:get_openai_api_key:79 - Retrieving API key for service: google
2026-07-14 18:50:05.223 | DEBUG    | autovqa.augment.config:get_openai_api_key:88 - Found API key in environment variable: GOOGLE_API_KEY
2026-07-14 18:50:05.223 | DEBUG    | autovqa.augment.config:get_openai_generation_settings:158 - Retrieving generation settings for service: google
2026-07-14 18:50:05.224 | DEBUG    | autovqa.augment.config:get_openai_generation_settings:163 - Service 'google' not found in config file, ignoring settings.It is advised to set generation settings in the config file,such as model_name, temperature, max_tokens: in the [{service_name}] section.
2026-07-14 18:50:11.102 | DEBUG    | autovqa.augment.client:generate_response:108 - Response generated.
2026-07-14 18:50:11.102 | DEBUG    | autovqa.augment.client:generate_response:109 - Remained samples per level: [2, 1, 2, 2, 2]


Generated 1 QA entries -> ./output/augmented_vqa.json


/home/ddyln/miniconda3/envs/autovqa/lib/python3.11/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=VQA(question='Con mèo đ...g gian', reason_depth=2), input_type=VQA])
  return self.__pydantic_serializer__.to_python(


---
## Step 4: Evaluate

Score each QA pair and its image across multiple dimensions:
- **Image quality** (clarity, occlusion, object density, etc.)
- **Image diversity** (scene type, main object, cultural context, etc.)
- **Text quality** (grammar, ambiguity, QA structure, etc.)
- **VQA correlation** (question↔image, answer↔image, reasoning depth)

### Prerequisites
Set the following environment variables (or use a `.env` file):

```env
GOOGLE_API_KEY=your_api_key
PROJECT_ID=your_gcp_project
LOCATION=us-central1
MODEL_NAME=gemini-2.5-flash
```

Output: `./output/evaluated_vqa.json`

In [4]:
from autovqa.evaluate import run_evaluate

# Run on 1 sample to test
# Results are saved as checkpoints in src/autovqa/evaluate/checkpoints/
run_evaluate(
    text_path="data/data_text/Data/combined_dataset/datasetQA_combined.json",
    image_path="data/data_images/raw_images_from_urls",
    output_dir="output",
    limit_samples=1,
    max_retries=3,
)

Resume from checkpoint_1 with 1 entries
Processing image 1: http://images.cocodataset.org/train2017/000000224483.jpg
Results saved to output/checkpoints/raw_answers_checkpoint_2.json


---
## Steps 5–7: EDA → Filter → Balance

Run the analysis pipeline on the **evaluated** output from Step 4.


> The input JSON must contain the `etp_`, `eip_`, `idp_`, `vqac_` scored fields produced by the Evaluate step.

In [ ]:
import json
import os
from autovqa import eda_pipeline, filter_pipeline, balancer_pipeline


def process_vqa_data(
    data_path: str,
    output_dir: str = "./output",
    filter_threshold: float = 0.5,
    generate_reports: bool = True,
):
    """
    Run EDA → Filter → Balance on evaluated VQA data.

    Args:
        data_path: Path to evaluated JSON (output of run_evaluate)
        output_dir: Directory for outputs
        filter_threshold: Ratio of labels that must pass (0.0–1.0)
        generate_reports: Whether to generate EDA Excel reports

    Returns:
        pd.DataFrame: Final balanced DataFrame
    """
    os.makedirs(output_dir, exist_ok=True)

    # Step 5: Load evaluated data
    print("Step 5: Loading evaluated data...")
    with open(data_path, "r", encoding="utf-8") as f:
        data = json.load(f)[:1]  # limit to 1 record for this example
    print(f"  Loaded {len(data)} records")

    # Step 6: EDA Pipeline
    print("Step 6: Running EDA pipeline...")
    df = eda_pipeline(
        data=data,
        output_dir=os.path.join(output_dir, "reports"),
        generate_report=generate_reports,
    )
    print(f"  EDA complete: {len(df)} records")

    # Step 7: Filter Pipeline
    print("Step 7: Running filter pipeline...")
    df = filter_pipeline(
        data=df,
        threshold=filter_threshold,
        show_stats=True,
    )
    print(f"  Filtering complete: {len(df)} records")

    # Step 7: Balancer Pipeline
    print("Step 7: Running balancer pipeline...")
    df = balancer_pipeline(
        df_raw=df,
        output_path=os.path.join(output_dir, "balanced_data.csv"),
    )
    print(f"  Balancing complete: {len(df)} records")

    print("\nPipeline complete!")
    return df

In [6]:
# Run on the evaluated output
# NOTE: Update data_path to point to your evaluated checkpoint JSON if needed
df_final = process_vqa_data(
    # data_path="./output/evaluated_vqa.json",
    data_path="/home/ddyln/Documents/School/Code/AutoVQA/docs/data/text/text/combined_evaluation_data.json",
    output_dir="./output",
    filter_threshold=0.5,
    generate_reports=True,
)

Step 4: Loading evaluated data...


2026-07-14 17:57:56.908 | INFO     | autovqa.eda.eda:run:360 - Running EDA pipeline...
2026-07-14 17:57:56.928 | INFO     | autovqa.eda.eda:get_report_on_data:235 - Generating detailed statistics report...
2026-07-14 17:57:56.957 | INFO     | autovqa.eda.eda:get_report_on_data:254 - Writing reports...
2026-07-14 17:57:56.957 | INFO     | autovqa.eda.eda:get_report_on_data:268 -   ✗ Main statistics: No module named 'openpyxl'
2026-07-14 17:57:56.960 | INFO     | autovqa.eda.eda:get_report_on_data:274 -   ✗ Scene types: No module named 'openpyxl'
2026-07-14 17:57:56.961 | INFO     | autovqa.eda.eda:get_report_on_data:280 -   ✗ Main objects: No module named 'openpyxl'
2026-07-14 17:57:56.961 | INFO     | autovqa.eda.eda:run:374 - EDA pipeline completed.
2026-07-14 17:57:56.963 | INFO     | autovqa.filter.filter:filter:61 - Records to keep: 1 / 1 (100.00%)
2026-07-14 17:57:56.964 | INFO     | autovqa.filter.filter:filter:78 - Filtered 1 rows to 1 rows (threshold: 0.5)
2026-07-14 17:57:56.9

  Loaded 1 records
Step 5: Running EDA pipeline...
  EDA complete: 1 records
Step 6: Running filter pipeline...
  Filtering complete: 1 records
Step 7: Running balancer pipeline...
  Balancing complete: 1 records

Pipeline complete!


In [7]:
# Save final results
df_final.to_csv("./output/final_dataset.csv", index=False)
df_final.to_json("./output/final_dataset.json", orient="records", force_ascii=False, indent=2)
print(f"Saved {len(df_final)} records to ./output/")

Saved 1 records to ./output/
